# L05 · Learning from Experience: MC, TD, and Q-learning

## Goal

- compare MC and TD targets
- compute an off-policy target
- separate termination from truncation

## Setup

This cell fixes CPU, seed, offline status, and the split hash first. Toy code uses deterministic CPU operations; package trainers retain their strict global default.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L05:toy:42").hexdigest()
print(f"lesson=L05 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L05 language=en profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.12.13 rl_study=0.1.0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:c7ad33c1f4e04332bf1b1303825d926c1144f26e9eddc8977bf61b78dda851de data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. Position and core equation

⏱ 5 min · 1/3 section · [CORE]

Position: MDP/Bellman → **MC, TD, and Q-learning** → DQN

$$G_t=R_{t+1}+\gamma G_{t+1},\qquad y_t^{TD}=R_{t+1}+\gamma(1-d_t)V(S_{t+1})$$

MC uses returns observed through episode end, giving low bias but high variance. TD bootstraps from the next value, learning earlier while inheriting estimation error. Q-learning is off-policy because its target uses the max-Q action rather than the action actually taken.

### 2. Run with small numbers

⏱ 6 min · 2/3 section · [CORE]

**Predict first:** Does a next-state value of 99 enter the TD target on a terminal transition? Write an answer for 20 seconds, then run the cell.

<details><summary>Show answer</summary>No. `(1-d_t)` zeros the bootstrap, so the target is the final reward 1.</details>

In [2]:
from rl_study.algorithms.tabular import (
    monte_carlo_returns, q_learning_target, td_target
)
rewards = torch.tensor([-0.01, -0.01, 1.0])
mc_targets = monte_carlo_returns(rewards, gamma=0.9)
td_targets = td_target(
    rewards, torch.tensor([0.7, 0.8, 99.0]),
    torch.tensor([False, False, True]), gamma=0.9
)
q_target = q_learning_target(
    torch.tensor([1.0]), torch.tensor([[2.0, 4.0]]),
    torch.tensor([False]), gamma=0.9
)
print({"mc": mc_targets.tolist(), "td": td_targets.tolist(),
       "off_policy_q_target": float(q_target[0])})

{'mc': [0.7909999489784241, 0.8899999856948853, 1.0], 'td': [0.6200000047683716, 0.7099999785423279, 1.0], 'off_policy_q_target': 4.599999904632568}


### 3. Implementation anatomy

⏱ 6 min · 3/3 section · [DEEP DIVE]

**Why this implementation:** Putting all three targets on one trajectory changes one property at a time: variance, bootstrapping, and off-policy choice. Analytic values audit implementation boundaries before training curves do.

**Common trap:** Treating time-limit `truncated` as environment `terminated` discards valid bootstrap information. The API and tests keep the two flags separate. Regression tests: `test_q_terminal_target`, `test_q_off_policy_target`.

**Checkpoint:** Continue when you can explain just one printed value.

## Checks

In [3]:
assert td_targets[-1].item() == 1.0
assert torch.allclose(q_target, torch.tensor([4.6]))
print("checks=passed")

checks=passed


**Recall:** Which target, MC or TD, directly depends on the current value estimate? Answer in one or two sentences.

## Mistakes I Revisit

- Assuming a finite loss proves the implementation is correct.
- Merging `terminated` with `truncated`, or prompt with action.
- Turning one tiny seed into an algorithm ranking.

## 60-Second Recap

- **Run conclusion:** The terminal TD target stays at 1.0, while Q-learning uses max next-Q of 4 to produce 4.6.
- Executable checks: `test_q_terminal_target`, `test_q_off_policy_target`.
- The output is a fixed-seed toy run, not a paper-scale result.

## Next Steps

1. L06 replaces the Q table with a neural network and studies target networks and replay.
2. Break one `[CORE]` assertion and read the failure.
3. Open the package test and connect the notebook equation to its production guard.

[Implementation note](../../docs/algorithms/classic.md) · [Course map](../../docs/course-map.en.md)

## Sources

- `sutton-barto-rl2` — `docs/sources.yml`